In [1]:
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

MODEL_ID = "./cifar100-vit"  # Or "username/model-repo"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the preprocessing configuration and fine-tuned weights
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(MODEL_ID)

model = model.to(device)
model.eval()

print("Device:", device)
print("Number of classes:", model.config.num_labels)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Device: cuda
Number of classes: 100


In [2]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import CIFAR100

# Convert batches of PIL images using the model's saved processor
def collate_fn(batch):
    images, labels = zip(*batch)

    inputs = processor(
        images=list(images),
        return_tensors="pt"
    )

    pixel_values = inputs["pixel_values"]
    labels = torch.tensor(labels, dtype=torch.long)

    return pixel_values, labels


@torch.inference_mode()
def evaluate(model, dataset, batch_size=64):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn
    )

    model.eval()
    correct = 0
    total = 0

    for pixel_values, labels in loader:
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)

        logits = model(pixel_values=pixel_values).logits
        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return 100.0 * correct / total

In [3]:
clean_test = CIFAR100(
    root="./data",
    train=False,
    download=True
)

clean_accuracy = evaluate(model, clean_test)

print(f"CIFAR-100 Test Accuracy: {clean_accuracy:.2f}%")

100%|██████████| 169M/169M [46:00<00:00, 61.2kB/s] 


CIFAR-100 Test Accuracy: 92.62%


In [3]:
from pathlib import Path
import pandas as pd

C_ROOT = Path("./data/CIFAR-100-C")

labels = np.load(C_ROOT / "labels.npy")


In [4]:

# Automatically find all corruption files
corruption_files = sorted(
    p for p in C_ROOT.glob("*.npy")
    
    if p.name != "labels.npy"
)


class CIFAR100CArrayDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.fromarray(np.asarray(self.images[idx]))
        label = int(self.labels[idx])
        return image, label

In [7]:
results = []

for corruption_file in corruption_files:
    corruption = corruption_file.stem

    # Memory-map the array to avoid loading the entire file into RAM
    images = np.load(corruption_file, mmap_mode="r")

    for severity in range(1, 6):
        start = (severity - 1) * 10_000
        end = severity * 10_000

        severity_images = images[start:end]

        # Usually labels.npy contains 10,000 labels reused at each severity.
        # This also supports datasets where it contains 50,000 labels.
        if len(labels) == len(images):
            severity_labels = labels[start:end]
        else:
            severity_labels = labels

        dataset = CIFAR100CArrayDataset(
            severity_images,
            severity_labels
        )

        accuracy = evaluate(model, dataset, batch_size=64)

        results.append({
            "corruption": corruption,
            "severity": severity,
            "accuracy": accuracy
        })

        print(
            f"{corruption:20s} | "
            f"Severity {severity} | "
            f"Accuracy: {accuracy:.2f}%"
        )

    del images

brightness           | Severity 1 | Accuracy: 92.53%
brightness           | Severity 2 | Accuracy: 92.47%
brightness           | Severity 3 | Accuracy: 91.61%
brightness           | Severity 4 | Accuracy: 90.63%
brightness           | Severity 5 | Accuracy: 87.67%
contrast             | Severity 1 | Accuracy: 92.19%
contrast             | Severity 2 | Accuracy: 90.52%
contrast             | Severity 3 | Accuracy: 88.92%
contrast             | Severity 4 | Accuracy: 84.89%
contrast             | Severity 5 | Accuracy: 63.69%
defocus_blur         | Severity 1 | Accuracy: 92.39%
defocus_blur         | Severity 2 | Accuracy: 91.80%
defocus_blur         | Severity 3 | Accuracy: 90.51%
defocus_blur         | Severity 4 | Accuracy: 87.82%
defocus_blur         | Severity 5 | Accuracy: 82.05%
elastic_transform    | Severity 1 | Accuracy: 88.99%
elastic_transform    | Severity 2 | Accuracy: 89.24%
elastic_transform    | Severity 3 | Accuracy: 87.01%
elastic_transform    | Severity 4 | Accuracy: 

In [8]:
df = pd.DataFrame(results)

# Rows = corruption types; columns = severity levels
accuracy_table = df.pivot(
    index="corruption",
    columns="severity",
    values="accuracy"
)

print("\nAccuracy (%) by corruption and severity:")
print(accuracy_table.round(2))

print("\nMean accuracy at each severity:")
print(df.groupby("severity")["accuracy"].mean().round(2))

print(
    f"\nMean CIFAR-100-C accuracy: "
    f"{df['accuracy'].mean():.2f}%"
)


Accuracy (%) by corruption and severity:
severity               1      2      3      4      5
corruption                                          
brightness         92.53  92.47  91.61  90.63  87.67
contrast           92.19  90.52  88.92  84.89  63.69
defocus_blur       92.39  91.80  90.51  87.82  82.05
elastic_transform  88.99  89.24  87.01  78.75  65.83
fog                92.06  89.81  87.68  83.06  67.81
frost              89.71  87.29  82.24  80.99  75.60
gaussian_blur      92.36  90.32  88.35  86.15  80.13
gaussian_noise     81.47  69.86  56.85  51.34  44.99
glass_blur         63.74  64.23  64.36  45.04  44.01
impulse_noise      86.99  80.95  74.55  60.12  47.24
jpeg_compression   83.59  76.01  73.58  70.12  64.99
motion_blur        90.17  86.63  82.13  82.36  76.52
pixelate           90.85  86.94  85.73  74.94  55.31
saturate           87.24  80.72  91.65  87.79  82.28
shot_noise         85.71  79.94  64.49  58.19  46.81
snow               90.16  85.51  84.70  81.81  78.78
spat

In [6]:
C_ROOT = Path("./data/CIFAR-100-C")

images = np.load(C_ROOT / "gaussian_noise.npy")
labels = np.load(C_ROOT / "labels.npy")

print(images.shape)  # (50000, 32, 32, 3)
print(labels.shape)  # Typically (10000,)

(50000, 32, 32, 3)
(50000,)


In [7]:
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset

# Load the file
images = np.load(C_ROOT / "gaussian_noise.npy", mmap_mode="r")
labels = np.load(C_ROOT / "labels.npy")

class GaussianNoiseDataset(Dataset):
    def __init__(self, images, labels, severity, indices):
        self.images = images
        self.labels = labels
        self.severity = severity
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        original_idx = self.indices[idx]
        array_idx = (self.severity - 1) * 10000 + original_idx

        image = Image.fromarray(np.array(self.images[array_idx]))
        label = int(self.labels[original_idx])

        return image, label


train_indices = np.arange(0, 8000)
heldout_indices = np.arange(8000, 10000)

# Training uses severity levels 1, 2, and 3
train_dataset = ConcatDataset([
    GaussianNoiseDataset(images, labels, s, train_indices)
    for s in [1, 2, 3]
])

# Validation uses held-out original images at severity levels 1-3
val_dataset = ConcatDataset([
    GaussianNoiseDataset(images, labels, s, heldout_indices)
    for s in [1, 2, 3]
])

print("Training samples:", len(train_dataset))  # 24000
print("Validation samples:", len(val_dataset))  # 6000

Training samples: 24000
Validation samples: 6000


In [8]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

def collate_fn(batch):
    images, labels = zip(*batch)

    pixel_values = processor(
        images=list(images),
        return_tensors="pt"
    )["pixel_values"]

    return pixel_values, torch.tensor(labels, dtype=torch.long)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

# Use a small learning rate to preserve pretrained features
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
criterion = CrossEntropyLoss()

def run_epoch(loader, training=False):
    model.train(training)
    total_loss = 0
    correct = 0
    total = 0

    context = torch.enable_grad() if training else torch.inference_mode()

    with context:
        for pixel_values, targets in loader:
            pixel_values = pixel_values.to(device)
            targets = targets.to(device)

            if training:
                optimizer.zero_grad()

            logits = model(pixel_values=pixel_values).logits
            loss = criterion(logits, targets)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * targets.size(0)
            correct += (logits.argmax(dim=1) == targets).sum().item()
            total += targets.size(0)

    return total_loss / total, 100 * correct / total


best_val_acc = 0

for epoch in range(5):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader)

    print(
        f"Epoch {epoch + 1}/5 | "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc:.2f}% | "
        f"Val acc: {val_acc:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained("./vit-gaussian-noise")
        processor.save_pretrained("./vit-gaussian-noise")

print("Best validation accuracy:", best_val_acc)

Epoch 1/5 | Train loss: 0.5867 | Train acc: 84.45% | Val acc: 84.45%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2/5 | Train loss: 0.2356 | Train acc: 93.55% | Val acc: 84.97%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 3/5 | Train loss: 0.1031 | Train acc: 97.46% | Val acc: 85.12%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 4/5 | Train loss: 0.0484 | Train acc: 99.02% | Val acc: 85.15%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 5/5 | Train loss: 0.0229 | Train acc: 99.69% | Val acc: 85.33%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best validation accuracy: 85.33333333333333


In [9]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    "./vit-gaussian-noise"
).to(device)

for severity in [4, 5]:
    test_dataset = GaussianNoiseDataset(
        images, labels, severity, heldout_indices
    )

    test_loss, test_acc = run_epoch(
        DataLoader(
            test_dataset,
            batch_size=64,
            shuffle=False,
            num_workers=0,
            collate_fn=collate_fn
        ),
        training=False
    )

    print(f"Gaussian noise severity {severity}: {test_acc:.2f}%")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Gaussian noise severity 4: 78.65%
Gaussian noise severity 5: 76.00%
